# SHA-512 Compression Function

The Secure Hash Algorithms are specified in:

[NIST Secure Hash Standard (SHS) FIPS 180-4](https://doi.org/10.6028/NIST.FIPS.180-4)  
Hash algorithms used to detect whether messages have been changed.

In [1]:
# Numerical arrays and types.
import numpy as np

# Official Python Hashing module.
import hashlib

## Setup

SHA-512 generates a series of hash values, one for each 1024-bit block of the padded message.  
The compression function $C$ generates the next hash value from the previous hash and the next block.  
The final hash value (after the final block) is the message digest [(SHS, p.3)](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf).  

$C: (H, M) \rightarrow H': \{0, 1\}^{512} \times \{0, 1\}^{1024} \rightarrow \{0, 1\}^{512}$

Note the output set $\{0, 1\}^{512}$ is smaller than the input set $\{0, 1\}^{512} \times \{0, 1\}^{1024}$.  
So, there are distinct inputs that give the same output - called collisions.

### Types

SHA-512 centres on 64-bit words [(SHS, p.3)](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf).

| Type  |  Size    | Python Type |
|-------|----------|-------------|
| Word  | 64 Bits  | Single [`numpy.uint64`](https://numpy.org/doc/stable/reference/arrays.scalars.html#numpy.uint64) |
| Hash  | 8 Words  | Length 8 [`numpy.array`](https://numpy.org/doc/stable/reference/generated/numpy.array.html) with [`dtype`](https://numpy.org/doc/stable/reference/arrays.dtypes.html) [`numpy.uint64`](https://numpy.org/doc/stable/reference/arrays.scalars.html#numpy.uint64) |
| Block | 16 Words | Length 16 [`numpy.array`](https://numpy.org/doc/stable/reference/generated/numpy.array.html) with [`dtype`](https://numpy.org/doc/stable/reference/arrays.dtypes.html) [`numpy.uint64`](https://numpy.org/doc/stable/reference/arrays.scalars.html#numpy.uint64) |

### Overflow

All additions are performed modulo $2^{64}$  [(SHS, p.5)](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf), which corresponds to [`uint64` overflow](https://numpy.org/doc/stable/user/basics.types.html#overflow-errors).

In [2]:
# Largest possible uint64.
allones = np.uint64(0xFFFFFFFFFFFFFFFF)

# One as a uint64.
one = np.uint64(1)

In [3]:
# Generates a warning.
wrapped = allones + one

# Show.
print(f'{wrapped:064b}')

0000000000000000000000000000000000000000000000000000000000000000


/var/folders/0c/3629hfbs2w15_4mtn1knljhw0000gn/T/ipykernel_93695/2440023869.py:2: RuntimeWarning: overflow encountered in scalar add
  wrapped = allones + one


In [4]:
# Ignore warnings.
with np.errstate(over="ignore"):
    wrapped = allones + one

In [5]:
# Show the overflow works.
print(f"All Ones: 0x{int(allones):016x}")
print(f"Plus One: 0x{int(wrapped):016x}")

# Test the overflow works.
assert wrapped == np.uint64(0)

All Ones: 0xffffffffffffffff
Plus One: 0x0000000000000000


## Bitwise Operators

Section 2.2.2 defines the bitwise operators used [(SHS, p.5)](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf).

| SHS        | Python | Name  | Explanation                                 |
|------------|--------|-------|---------------------------------------------|
| $\land$    | `&`    | `AND` | Only `1` where both are `1`.                |
| $\lor$     | `\|`   | `OR`  | Only `1` where either or both are `1`.      |
| $\oplus$   | `^`    | `XOR` | Only `1` where either but not both are `1`. |
| $\neg$     | `~`    | `NOT` | Only `1` where input is `0`.                |
| $\gg$       | `>>`   | `SHR` | Moves bits right, fills with `0`.           |
| $\ll$       | `<<`   | `SHL` | Moves bits left, fills with `0`.            |

In [6]:
# All combinations of two bits.
x = np.uint64(0b1100)
y = np.uint64(0b1010)

### AND

In [7]:
# AND.
print(f'  x: {x:04b}')
print(f'  y: {y:04b}')
print(f'x&y: {x&y:04b}')

  x: 1100
  y: 1010
x&y: 1000


### OR

In [8]:
# OR.
print(f'  x: {x:04b}')
print(f'  y: {y:04b}')
print(f'x|y: {x|y:04b}')

  x: 1100
  y: 1010
x|y: 1110


### XOR

In [9]:
# XOR.
print(f'  x: {x:04b}')
print(f'  y: {y:04b}')
print(f'x^y: {x^y:04b}')

  x: 1100
  y: 1010
x^y: 0110


### NOT

In [10]:
# NOT.
print(f'x:  {x:064b}')
print(f'~x: {~x:064b}')

x:  0000000000000000000000000000000000000000000000000000000000001100
~x: 1111111111111111111111111111111111111111111111111111111111110011


### Right Shift

In [11]:
# Right shift.
print(f'x:    {x:064b}')
print(f'x>>1: {x>>1:064b}')

x:    0000000000000000000000000000000000000000000000000000000000001100
x>>1: 0000000000000000000000000000000000000000000000000000000000000110


### Left Shift

In [12]:
# Left shift.
print(f'x:    {x:064b}')
print(f'x<<1: {x<<1:064b}')

x:    0000000000000000000000000000000000000000000000000000000000001100
x<<1: 0000000000000000000000000000000000000000000000000000000000011000


## Bitwise Functions

Section 2.2.2 also defines some functions built from the basic operations [(SHS, p.5)](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf).

### Shift versus Rotation

Right shift $\tt{SHR}^{n}(x)$ discards $n$ rightmost bits, inserts zeroes on the left.

Right rotation $\tt{ROTR}^{n}(x)$ wraps discarded bits to the left [(SHS, p.8)](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf).

$\tt{ROTR}^{n}(x) = (x \gg n) \; \lor \; (x \ll (64 - n) )$

Left shift and rotation are similar.

In [13]:
def shr(x, n):
    """Logical right shift of a 64-bit unsigned word."""
    return np.uint64(x) >> n

In [14]:
def rotr(x, n):
    """Right rotation of a 64-bit unsigned word."""
    x = np.uint64(x)
    n = n % 64
    return (x >> n) | (x << ((64 - n) % 64))

In [15]:
# Example input.
x = np.uint64(0x1234567890abcdef)

print(f"x:           {int(x):064b}")
print(f"SHR^2(x):    {int(shr(x, 2)):064b}")
print(f"ROTR^2(x):   {int(rotr(x, 2)):064b}")

assert shr(x, 4) == np.uint64(0x01234567890abcde)
assert rotr(x, 4) == np.uint64(0xf1234567890abcde)
assert rotr(x, 64) == x

x:           0001001000110100010101100111100010010000101010111100110111101111
SHR^2(x):    0000010010001101000101011001111000100100001010101111001101111011
ROTR^2(x):   1100010010001101000101011001111000100100001010101111001101111011


### Choose

$\tt{Ch}(x,y,z) = (x \land y) \oplus (\neg x \land z)$  
$\tt{Ch}$ selects a bit from $y$ where $x$ has a 1; otherwise it selects the corresponding bit from $z$ [(SHS, p.11)](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf).  

In [16]:
def ch(x, y, z):
    """Choose: take each bit from y or z according to x."""
    return (x & y) ^ (~x & z)

In [17]:
# All possible 3 bit combos.
x = np.uint64(0b11110000)
y = np.uint64(0b11001100)
z = np.uint64(0b10101010)

# Show.
print(f"x:  {x:08b}")
print(f"y:  {y:08b}")
print(f"z:  {z:08b}")
print(f"Ch: {ch(x, y, z):08b}")

# Test value.
assert ch(x, y, z) == np.uint64(0b11001010)

x:  11110000
y:  11001100
z:  10101010
Ch: 11001010


### Majority

$\tt{Maj}(x,y,z) = (x \land y) \oplus (x \land z) \oplus (y \land z)$  
$\tt{Maj}$ produces 1 if at least two of the three bits are 1, otherwise 0 [(SHS, p.11)](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf).  

In [18]:
def maj(x, y, z):
    """Majority of three words, bit by bit."""
    return (x & y) ^ (x & z) ^ (y & z)

In [19]:
# All possible 3 bit combos.
x = np.uint64(0b11110000)
y = np.uint64(0b11001100)
z = np.uint64(0b10101010)

# Show.
print(f"x:   {x:08b}")
print(f"y:   {y:08b}")
print(f"z:   {z:08b}")
print(f"Maj: {maj(x, y, z):08b}")

# Test value.
assert maj(x, y, z) == np.uint64(0b11101000)

x:   11110000
y:   11001100
z:   10101010
Maj: 11101000


### Sigma Functions

The symbol $\Sigma$ is an uppercase sigma in Greek.  
The symbol $\sigma$ is a lowercase sigma in Greek.  
The exponents are rotation/shift counts [(SHS p.11)](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf).  

$
\begin{aligned}
\Sigma_0(x)&=\tt{ROTR}^{28}(x) \oplus \tt{ROTR}^{34}(x) \oplus \tt{ROTR}^{39}(x),\\
\Sigma_1(x)&=\tt{ROTR}^{14}(x) \oplus \tt{ROTR}^{18}(x) \oplus \tt{ROTR}^{41}(x),\\
\sigma_0(x)&=\tt{ROTR}^{ 1}(x) \oplus \tt{ROTR}^{ 8}(x) \oplus \tt{SHR }^{ 7}(x),\\
\sigma_1(x)&=\tt{ROTR}^{19}(x) \oplus \tt{ROTR}^{61}(x) \oplus \tt{SHR }^{ 6}(x).
\end{aligned}
$

In [20]:
def Sigma0(x):
    """From the Secure Hash Standard, Page 11."""
    return rotr(x, 28) ^ rotr(x, 34) ^ rotr(x, 39)

In [21]:
def Sigma1(x):
    """From the Secure Hash Standard, Page 11."""
    return rotr(x, 14) ^ rotr(x, 18) ^ rotr(x, 41)

In [22]:
def sigma0(x):
    """From the Secure Hash Standard, Page 11."""
    return rotr(x, 1) ^ rotr(x, 8) ^ shr(x, 7)

In [23]:
def sigma1(x):
    """From the Secure Hash Standard, Page 11."""
    return rotr(x, 19) ^ rotr(x, 61) ^ shr(x, 6)

In [24]:
# All possible nibbles.
x = np.uint64(0x0123456789ABCDEF)

print(f"Input:  {x:016x}")
print(f"Sigma0: {Sigma0(x):016x}")
print(f"Sigma1: {Sigma1(x):016x}")
print(f"sigma0: {sigma0(x):016x}")
print(f"sigma1: {sigma1(x):016x}")

Input:  0123456789abcdef
Sigma0: b7c57a100c7ec1ab
Sigma1: 7703112333475567
sigma0: 6f92c77c6c4f1aa1
sigma1: 70a3460dbbd4317a


In [25]:
# NIST's complete, already-padded, one-block SHA-512 input for b"abc".
abcblock = np.array(
    (0x6162638000000000,) + (0x0000000000000000,) * 14 + (0x0000000000000018,),
    dtype=np.uint64,
)

# Check block is 16*64=1024 bits.
assert len(abcblock) == 16

# Print the block in 64-bit words.
for i, word in enumerate(abcblock):
    print(f"M[{i:2d}] = {word:016x}")

M[ 0] = 6162638000000000
M[ 1] = 0000000000000000
M[ 2] = 0000000000000000
M[ 3] = 0000000000000000
M[ 4] = 0000000000000000
M[ 5] = 0000000000000000
M[ 6] = 0000000000000000
M[ 7] = 0000000000000000
M[ 8] = 0000000000000000
M[ 9] = 0000000000000000
M[10] = 0000000000000000
M[11] = 0000000000000000
M[12] = 0000000000000000
M[13] = 0000000000000000
M[14] = 0000000000000000
M[15] = 0000000000000018


## Constants

### Iteration Constants

Each round uses one 64-bit constant $K_t$.  

These are the fractional parts of the cube roots of the first 80 primes.  

We hard code the from the standard [(SHS, p.12)](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf).

In [26]:
# From Section 4.2.3 of the Secure Hash Standard.
K = np.array((
    0x428a2f98d728ae22, 0x7137449123ef65cd, 0xb5c0fbcfec4d3b2f, 0xe9b5dba58189dbbc,
    0x3956c25bf348b538, 0x59f111f1b605d019, 0x923f82a4af194f9b, 0xab1c5ed5da6d8118,
    0xd807aa98a3030242, 0x12835b0145706fbe, 0x243185be4ee4b28c, 0x550c7dc3d5ffb4e2,
    0x72be5d74f27b896f, 0x80deb1fe3b1696b1, 0x9bdc06a725c71235, 0xc19bf174cf692694,
    0xe49b69c19ef14ad2, 0xefbe4786384f25e3, 0x0fc19dc68b8cd5b5, 0x240ca1cc77ac9c65,
    0x2de92c6f592b0275, 0x4a7484aa6ea6e483, 0x5cb0a9dcbd41fbd4, 0x76f988da831153b5,
    0x983e5152ee66dfab, 0xa831c66d2db43210, 0xb00327c898fb213f, 0xbf597fc7beef0ee4,
    0xc6e00bf33da88fc2, 0xd5a79147930aa725, 0x06ca6351e003826f, 0x142929670a0e6e70,
    0x27b70a8546d22ffc, 0x2e1b21385c26c926, 0x4d2c6dfc5ac42aed, 0x53380d139d95b3df,
    0x650a73548baf63de, 0x766a0abb3c77b2a8, 0x81c2c92e47edaee6, 0x92722c851482353b,
    0xa2bfe8a14cf10364, 0xa81a664bbc423001, 0xc24b8b70d0f89791, 0xc76c51a30654be30,
    0xd192e819d6ef5218, 0xd69906245565a910, 0xf40e35855771202a, 0x106aa07032bbd1b8,
    0x19a4c116b8d2d0c8, 0x1e376c085141ab53, 0x2748774cdf8eeb99, 0x34b0bcb5e19b48a8,
    0x391c0cb3c5c95a63, 0x4ed8aa4ae3418acb, 0x5b9cca4f7763e373, 0x682e6ff3d6b2b8a3,
    0x748f82ee5defb2fc, 0x78a5636f43172f60, 0x84c87814a1f0ab72, 0x8cc702081a6439ec,
    0x90befffa23631e28, 0xa4506cebde82bde9, 0xbef9a3f7b2c67915, 0xc67178f2e372532b,
    0xca273eceea26619c, 0xd186b8c721c0c207, 0xeada7dd6cde0eb1e, 0xf57d4f7fee6ed178,
    0x06f067aa72176fba, 0x0a637dc5a2c898a6, 0x113f9804bef90dae, 0x1b710b35131c471b,
    0x28db77f523047d84, 0x32caab7b40c72493, 0x3c9ebe0a15c9bebc, 0x431d67c49c100d4c,
    0x4cc5d4becb3e42b6, 0x597f299cfc657e2a, 0x5fcb6fab3ad6faec, 0x6c44198c4a475817,
), dtype=np.uint64)

# Check length.
assert K.shape == (80,)
# Check first.
assert K[0]  == np.uint64(0x428A2F98D728AE22)
# Check last.
assert K[79] == np.uint64(0x6C44198C4A475817)

# Show first and last.
print(f"K[0]  = {K[0]:016x}")
print(f"K[79] = {K[79]:016x}")

K[0]  = 428a2f98d728ae22
K[79] = 6c44198c4a475817


### Initial Hash

The initial hash is specified in the standard [(SHS p.15)](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf).  

$H^{(0)} = ( H_0^{(0)}, \ldots, H_7^{(0)} )$

They are the fractional parts of the square roots of the first eight primes.  

In [27]:
# From Section 5.3.5 of the Secure Hash Standard.
H0 = np.array([
    0x6a09e667f3bcc908,
    0xbb67ae8584caa73b,
    0x3c6ef372fe94f82b,
    0xa54ff53a5f1d36f1,
    0x510e527fade682d1,
    0x9b05688c2b3e6c1f,
    0x1f83d9abfb41bd6b,
    0x5be0cd19137e2179,
], dtype=np.uint64)

# Print as words.
for i, h in enumerate(H0):
    print(f"H0[{i}] = {h:016x}")

# Test length.
assert H0.shape == (8,)

H0[0] = 6a09e667f3bcc908
H0[1] = bb67ae8584caa73b
H0[2] = 3c6ef372fe94f82b
H0[3] = a54ff53a5f1d36f1
H0[4] = 510e527fade682d1
H0[5] = 9b05688c2b3e6c1f
H0[6] = 1f83d9abfb41bd6b
H0[7] = 5be0cd19137e2179


## Schedule

The first 16 schedule words are just the block words [(SHS p.24)](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf):

$ W_t = M_t, \qquad 0 \leq t \leq 15$

The remaining words are constructed `sigma0` and `sigma1` [(SHS p.24)](https://nvlpubs.nist.gov/nistpubs/FIPS/NIST.FIPS.180-4.pdf):

$W_t=\sigma_1(W_{t-2}) + W_{t-7} + \sigma_0(W_{t-15}) + W_{t-16}, \quad 16 \leq t \leq 79$

Remember all addition is modulo $2^{64}$.

In [28]:
def messageschedule(block):
    """Expand sixteen uint64 block words into 80 schedule words."""
    if len(block) != 16:
        raise ValueError("A SHA-512 block must contain 16 words.")

    # Initialise schedule.
    W = np.zeros(80, dtype=np.uint64)
    # First 16 words are directly from block.
    W[:16] = block

    # Ignore overflow warnings by design.
    with np.errstate(over="ignore"):
        # The rest of the words are calculated from the first 16.
        for t in range(16, 80):
            W[t] = sigma1(W[t - 2]) + W[t - 7] + sigma0(W[t - 15]) + W[t - 16]
    return W

### Schedule of `abc`

In [29]:
# Calculcate the message schedule.
W = messageschedule(abcblock)

# Show a selection of blocks.
for i in [0, 1, 14, 15, 16, 17, 18, 79]:
    print(f"W[{i:2d}] = {W[i]:016x}")

# Test length.
assert len(W) == 80
# Test first 16 words are original block.
assert np.all(W[:16] == abcblock)

W[ 0] = 6162638000000000
W[ 1] = 0000000000000000
W[14] = 0000000000000000
W[15] = 0000000000000018
W[16] = 6162638000000000
W[17] = 00030000000000c0
W[18] = 0a9699a24c700003
W[79] = 92aeeed1a7bcf7d2


## Compression Rounds

The eight input hash words go into working variables $(a,b,c,d,e,f,g,h)$.  

At round $t$, first calculate:

$
\begin{aligned}
T_1 &= h + \Sigma_1(e) + \operatorname{Ch}(e,f,g) + K_t + W_t \\
T_2 &= \Sigma_0(a) + \operatorname{Maj}(a,b,c).
\end{aligned}
$

Then simultaneously update the working variables (RHS values are old values, use tuple assignment):

$(a,b,c,d,e,f,g,h) \leftarrow (T_1 + T_2,\ a,\ b,\ c,\ d + T_1,\ e,\ f,\ g)$

Remember all addition is modulo $2^{64}$.

In [30]:
def sha512round(state, word, constant):
    """Apply one SHA-512 round to eight uint64 working words."""
    # Extract words from state.
    a, b, c, d, e, f, g, h = state

    # Ignore overflow warnings by design.
    with np.errstate(over="ignore"):
        # Temporary variables.
        T1 = h + Sigma1(e) + ch(e, f, g) + constant + word
        T2 = Sigma0(a) + maj(a, b, c)
        # Update working variables.
        return (T1 + T2, a, b, c, d + T1, e, f, g)

### Test First Round

The NIST `abc` sample gives the state after round $t=0$ [(NIST Examples, p.1)](https://csrc.nist.gov/CSRC/media/Projects/Cryptographic-Standards-and-Guidelines/documents/examples/SHA512.pdf).  

In [31]:
# Calculate worknig variables after first round.
afterround0 = sha512round(H0, W[0], K[0])

# Print new a and e.
print("New a:", f"{afterround0[0]:016x}")
print("New e:", f"{afterround0[4]:016x}")

# Tests.
assert afterround0[0] == np.uint64(0xF6AFCEB8BCFCDDF5)
assert afterround0[4] == np.uint64(0x58CB02347AB51F91)
assert afterround0[1] == H0[0]  # b takes the old a
assert afterround0[5] == H0[4]  # f takes the old e

New a: f6afceb8bcfcddf5
New e: 58cb02347ab51f91


### Last Step

After 80 rounds, the original eight state words are added back, componentwise, modulo $2^{64}$.

$H'_j = H_j + V_j, \qquad 0 \leq j<8$

## Complete Compression Function

The compression function takes the previous hash state and the next block.  
It returns the next eight-word hash value.

In [32]:
def sha512compress(state, block):
    """Compress one 1024-bit block using an eight-word SHA-512 state.

    Both inputs contain uint64 words. The output is eight uint64 words.
    Block preparation and padding are assumed to already be done.
    """
    if len(state) != 8:
        raise ValueError("The state must contain eight words.")

    words = messageschedule(block)
    working = tuple(np.uint64(x) for x in state)

    for t in range(80):
        working = sha512round(working, words[t], K[t])

    # Add the original chaining state back, modulo 2^64.
    with np.errstate(over="ignore"):
        return np.array(
            [np.uint64(x) + y for x, y in zip(state, working)],
            dtype=np.uint64,
        )

## The `abc` Test

In UTF-8, `"abc"` is `0x616263` in hexadecimal bytes - 24 bits.  
SHA-512 padding appends a single 1 bit and then any necessary zero bits.  
Given the input is always in bytes, appending a 1 bit means appending `0x80`.  
The original message length is then appended as a 128-bit big-endian integer.  

If the original length is $\ell$, the zero-padding count $k$ satisfies:  

$\ell + 1 + k \equiv (1024 - 128) \equiv 896 \pmod{1024}, \qquad 0 \leq k < 1024$  

For `"abc"`, $\ell = 24$, so $k = 871$.  

The final 128-bit length field encodes the integer $24 = \mathtt{0x18}$.  
The resulting 1024-bit block is `0x6162638000...00018`.  
We hard-code it below as this notebook focuses on compression, not padding.

Since `abc` occupies exactly one padded block, the first call gives the final digest.

$H^{(1)} = C(H^{(0)}, M^{(1)})$

We compare it with the published NIST digest, hard-coded independently of our implementation [(NIST Examples, p.4)](https://csrc.nist.gov/CSRC/media/Projects/Cryptographic-Standards-and-Guidelines/documents/examples/SHA512.pdf).  
We also compare against [Python's `hashlib`](https://docs.python.org/3/library/hashlib.html).

In [33]:
# We use a string to ensure the types don't do anything funny.
# Juxtaposed string literals are concatenated in Python.
expected = (
    "ddaf35a193617abacc417349ae204131"
    "12e6fa4e89a97ea20a9eeee64b55d39a"
    "2192992a274fc1a836ba3c23a3feebbd"
    "454d4423643ce80e2a9ac94fa54ca49f"
)

# Compress the single abcblock with the initial hash, giving the final digest.
abcresult = sha512compress(H0, abcblock)
# To string using Python strings' join method.
abcdigest = "".join(f"{word:016x}" for word in abcresult)

# Print our result, the expected result, and Python's hashlib result.
print("Our compression:", abcdigest)
print("NIST reference: ", expected)
print("Python hashlib: ", hashlib.sha512(b"abc").hexdigest())

# Check they're all equal.
assert abcdigest == expected
assert abcdigest == hashlib.sha512(b"abc").hexdigest()

Our compression: ddaf35a193617abacc417349ae20413112e6fa4e89a97ea20a9eeee64b55d39a2192992a274fc1a836ba3c23a3feebbd454d4423643ce80e2a9ac94fa54ca49f
NIST reference:  ddaf35a193617abacc417349ae20413112e6fa4e89a97ea20a9eeee64b55d39a2192992a274fc1a836ba3c23a3feebbd454d4423643ce80e2a9ac94fa54ca49f
Python hashlib:  ddaf35a193617abacc417349ae20413112e6fa4e89a97ea20a9eeee64b55d39a2192992a274fc1a836ba3c23a3feebbd454d4423643ce80e2a9ac94fa54ca49f


### Other Tests

Let's check that sha512compress does not modify it's inputs.

In [34]:
# Copy (deep) the initial hash.
originalstate = H0.copy()
# Copy the input block.
originalblock = abcblock.copy()
# Run the compression function.
out = sha512compress(H0, abcblock)

# Check the initial hash value did not change.
assert np.array_equal(H0, originalstate)
# Check the block value did not change.
assert np.array_equal(abcblock, originalblock)

In [35]:
# Test the compression functions reject wrong-sized inputs.
try:
    sha512compress(H0, abcblock[:15])
except ValueError:
    print("PASS: a 15-word block was rejected.")
else:
    raise AssertionError("Expected an error for a short block.")

PASS: a 15-word block was rejected.


## End